In [1]:
import os
import sqlite3
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split

# In notebooks, __file__ is not defined. Use the current working directory.
CWD = Path.cwd()
BASE_DIR = CWD.parent if CWD.name == "notebooks" else CWD
DB_PATH = BASE_DIR / "codecity.db"
MODELS_DIR = BASE_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Using DB:", DB_PATH)
print("Saving models to:", MODELS_DIR)

conn = sqlite3.connect(str(DB_PATH))
files_df = pd.read_sql_query(
    "SELECT snapshot_id, path, name, size, complexity, width, depth, height, is_test_file, area, aspect_ratio FROM files",
    conn,
)
conn.close()

# Simple synthetic label: mark top 20% complexity files as high risk
threshold = files_df["complexity"].quantile(0.8)
files_df["is_high_risk"] = (files_df["complexity"] >= threshold).astype(int)

feature_cols = [
    "size",
    "complexity",
    "width",
    "depth",
    "height",
    "is_test_file",
    "area",
    "aspect_ratio",
]

X = files_df[feature_cols].values
y = files_df["is_high_risk"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    n_jobs=-1,
    random_state=42,
)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))

model_path = MODELS_DIR / "risk_model.joblib"
meta_path = MODELS_DIR / "risk_model_meta.json"

joblib.dump(clf, str(model_path))

with open(meta_path, "w") as f:
    json.dump({"feature_cols": feature_cols}, f, indent=2)

print("Saved model to", model_path)
print("Saved metadata to", meta_path)

Using DB: c:\Users\LENOVO\OneDrive\Desktop\CodeCity\CodeCity\codecity.db
Saving models to: c:\Users\LENOVO\OneDrive\Desktop\CodeCity\CodeCity\models
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        16
           1       1.00      1.00      1.00         4

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20

ROC AUC: 1.0
Saved model to c:\Users\LENOVO\OneDrive\Desktop\CodeCity\CodeCity\models\risk_model.joblib
Saved metadata to c:\Users\LENOVO\OneDrive\Desktop\CodeCity\CodeCity\models\risk_model_meta.json
